# Notebook 03 — Join & Explore

**Goal:** join the four cleaned tables into one fact table (one row per validation) and do a first exploratory pass.

Reads `data/interim/*_clean.parquet` (written by notebook 02). Writes `data/interim/fact_validations.parquet`, the input to notebook 04.

Run the notebook from top to bottom. Do not run cells out of order.

In [14]:
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

validations = pd.read_parquet(INTERIM_DIR / "validations_clean.parquet")
routes = pd.read_parquet(INTERIM_DIR / "routes_clean.parquet")
vehicles = pd.read_parquet(INTERIM_DIR / "vehicles_clean.parquet")
stops = pd.read_parquet(INTERIM_DIR / "stops_clean.parquet")

print("Validations:", validations.shape)
print("Routes:", routes.shape)
print("Vehicles:", vehicles.shape)
print("Stops:", stops.shape)

Validations: (79149, 21)
Routes: (60, 6)
Vehicles: (180, 6)
Stops: (400, 5)


## Join strategy

Validations → routes → vehicles → stops with `left` joins and `validate="many_to_one"` (one validation joins to exactly one route/vehicle/stop). The route's district and the stop's district are both called `district` in their source tables, so each is renamed right after its own join (`route_district`, `stop_district`) — before the next join runs — so no join ever produces a pandas `_x`/`_y` suffix collision.

In [2]:
fact = validations.merge(
    routes[["route_id", "route_name", "mode", "district", "route_length_km"]],
    on="route_id",
    how="left",
    validate="many_to_one",
)
fact = fact.rename(columns={"district": "route_district"})

assert len(fact) == len(validations), "Route join changed the row count (fan-out)."
assert fact["route_name"].notna().all(), "Some validations did not match a route."

print("After route join:", fact.shape)

After route join: (79149, 25)


In [3]:
fact = fact.merge(
    vehicles[["vehicle_id", "vehicle_type", "depot", "vehicle_age_band", "capacity"]],
    on="vehicle_id",
    how="left",
    validate="many_to_one",
)

assert len(fact) == len(validations), "Vehicle join changed the row count (fan-out)."
assert fact["vehicle_type"].notna().all(), "Some validations did not match a vehicle."

print("After vehicle join:", fact.shape)

After vehicle join: (79149, 29)


In [4]:
fact = fact.merge(
    stops[["stop_id", "stop_name", "district", "zone", "has_shelter"]],
    on="stop_id",
    how="left",
    validate="many_to_one",
)
fact = fact.rename(columns={"district": "stop_district"})

assert len(fact) == len(validations), "Stop join changed the row count (fan-out)."
assert fact["stop_name"].notna().all(), "Some validations did not match a stop."

print("After stop join:", fact.shape)
print("Columns:", fact.columns.tolist())

After stop join: (79149, 33)
Columns: ['validation_id', 'validation_ts', 'route_id', 'vehicle_id', 'stop_id', 'card_id', 'fare_type', 'fare_amount', 'passenger_count', 'transfer_flag', 'is_reversed', 'date', 'year', 'month', 'year_month', 'quarter', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'revenue', 'route_name', 'mode', 'route_district', 'route_length_km', 'vehicle_type', 'depot', 'vehicle_age_band', 'capacity', 'stop_name', 'stop_district', 'zone', 'has_shelter']


## Load factor

Load factor is calculated per **trip**, defined as one vehicle on one route on one date in one clock hour: `trip boardings / vehicle capacity`. Boardings are summed (not counted) per trip so a reversed validation correctly cancels its original boarding, consistent with net boardings elsewhere in this project.

In [5]:
trip_keys = ["route_id", "vehicle_id", "date", "hour"]

trip_boardings = fact.groupby(trip_keys)["passenger_count"].transform("sum")
fact["load_factor"] = trip_boardings / fact["capacity"]

print("Trips:", fact.groupby(trip_keys).ngroups)
print("Average trip load factor:", round(fact["load_factor"].mean(), 4))
print(fact["load_factor"].describe())

Trips: 78741
Average trip load factor: 0.0097
count    79149.000000
mean         0.009652
std          0.006798
min         -0.050000
25%          0.004975
50%          0.008475
75%          0.014286
max          0.080645
Name: load_factor, dtype: float64


## Join integrity check

In [6]:
print("Rows:", len(fact))
assert len(fact) == 79149, "Unexpected row count after joining."

key_columns = [
    "route_name", "mode", "route_district", "route_length_km",
    "vehicle_type", "depot", "vehicle_age_band", "capacity",
    "stop_name", "stop_district", "zone", "has_shelter",
]
nulls = fact[key_columns].isna().sum()
assert nulls.sum() == 0, f"Unexpected nulls after joining:\n{nulls[nulls > 0]}"

print("No nulls introduced by the joins.")
print("Duplicate validation_id:", fact["validation_id"].duplicated().sum())

Rows: 79149
No nulls introduced by the joins.
Duplicate validation_id: 0


## Exploratory summaries

In [7]:
hourly_ridership = (
    fact.groupby(["hour", "is_weekend"])
    .agg(total_boardings=("passenger_count", "sum"))
    .reset_index()
)

print(hourly_ridership.head(10))

   hour  is_weekend  total_boardings
0     0       False              150
1     0        True               48
2     1       False              120
3     1        True               46
4     2       False              160
5     2        True               34
6     3       False              116
7     3        True               69
8     4       False              163
9     4        True               66


In [8]:
mode_ridership = (
    fact.groupby("mode")
    .agg(
        total_boardings=("passenger_count", "sum"),
        total_revenue=("revenue", "sum"),
        avg_load_factor=("load_factor", "mean"),
    )
    .round(4)
)

print(mode_ridership)

       total_boardings  total_revenue  avg_load_factor
mode                                                  
Bus              48211       88558.23           0.0135
Metro             8330       15244.33           0.0018
Tram             28706       52687.68           0.0055


In [9]:
fare_mix = (
    fact.groupby("fare_type")
    .agg(
        boardings=("passenger_count", "sum"),
        revenue=("revenue", "sum"),
    )
    .sort_values("boardings", ascending=False)
)

print(fare_mix)

             boardings    revenue
fare_type                        
Adult            47278  118196.62
Student          12719   19045.83
Senior           10113   12106.66
Pass Holder       8468     488.05
Child             6669    6653.08


In [10]:
top_routes = (
    fact.groupby(["route_id", "route_name"])
    .agg(total_boardings=("passenger_count", "sum"))
    .sort_values("total_boardings", ascending=False)
    .head(10)
)

print(top_routes)

                               total_boardings
route_id route_name                           
R032     Thomas Dam Line                  1506
R007     Robinson Radial Line             1505
R037     Eric Estate Line                 1495
R041     Erin Crescent Line               1495
R021     Martin Knoll Line                1479
R015     Andrea Village Line              1478
R051     Daniel Lake Line                 1477
R048     Eric Track Line                  1475
R049     Amber Tunnel Line                1469
R058     Jennifer Locks Line              1468


In [11]:
district_ridership = (
    fact.groupby("route_district")
    .agg(total_boardings=("passenger_count", "sum"))
    .sort_values("total_boardings", ascending=False)
)

print(district_ridership)

                total_boardings
route_district                 
Centro                    15796
Westend                   11346
Hillcrest                  8490
Old Town                   8475
Northgate                  8401
Parkview                   7074
Southbank                  5744
Eastfield                  5724
Harbourside                5687
Unknown                    4275
Riverside                  4235


## Headline numbers

A quick sanity check against the project's documented findings (see `README.md`).

In [12]:
print("Total boardings:", int(fact["passenger_count"].sum()))
print("Total revenue:", round(fact["revenue"].sum(), 2))
print("Unique cards (excl. SINGLE_RIDE):", fact.loc[fact["card_id"] != "SINGLE_RIDE", "card_id"].nunique())
print("Single-ride tickets:", (fact["card_id"] == "SINGLE_RIDE").sum())
print("Average trip load factor:", round(fact["load_factor"].mean(), 4))

Total boardings: 85247
Total revenue: 156490.24
Unique cards (excl. SINGLE_RIDE): 23460
Single-ride tickets: 9441
Average trip load factor: 0.0097


## Save the fact table

Parquet is the input notebook 04 reads. The CSV copies in `data/processed/` are kept for quick inspection and for the Power BI report.

In [13]:
fact.to_parquet(INTERIM_DIR / "fact_validations.parquet", index=False)

fact.to_csv(PROCESSED_DIR / "fact_table.csv", index=False, encoding="utf-8-sig")
validations.to_csv(PROCESSED_DIR / "validations_clean.csv", index=False, encoding="utf-8-sig")

print("Fact table saved:", fact.shape)
print("Saved to:", INTERIM_DIR / "fact_validations.parquet")

Fact table saved: (79149, 34)
Saved to: c:\Users\User\OneDrive\Desktop\kurs\CityTransit_capstone_FINAL_corrected\data\interim\fact_validations.parquet
